# A micrograd neuron, in SKaiNET

Port of the two-input single-neuron example from Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) — the canonical "see the autodiff graph" demo. All credit for the original tutorial and graph-visualisation style belongs to Karpathy; this notebook is just a tribute showing the same graph shape coming out of SKaiNET's DAG DSL through our notebook-side `asDot()` renderer.

Original snippet:

```python
from micrograd import nn
n = nn.Neuron(2)
x = [Value(1.0), Value(-2.0)]
y = n(x)
dot = draw_dot(y)
```

In [ ]:
USE {
    repositories {
        mavenCentral()
    }
    dependencies {
        implementation("sk.ainet.app:kotlin-notebook:0.25.0")
    }
}

## The translation

micrograd's `Neuron(2)` is `tanh(w · x + b)` over two inputs, with `w` random and `b` zero. SKaiNET 0.25.0 doesn't ship `tanh` as a DAG-DSL primitive, so the notebook integration provides a polyfill that composes the exact identity `tanh(x) = 2*sigmoid(2x) - 1` (see [`docs/upstream/tanh-activation.md`](../../docs/upstream/tanh-activation.md) for the upstream proposal that would replace it with a real primitive). The rendered graph shows the `mulScalar -> sigmoid -> mulScalar -> subScalar` decomposition rather than a single `tanh` block — faithful to what the kernel is actually computing today.

The leaf `Value(1.0)`/`Value(-2.0)` inputs become a `constant` tensor of shape `(1, 2)` so the values show up directly on the graph, the same way Karpathy's `draw_dot` surfaces the leaves. Returning the program from a cell renders the graph as inline SVG via the bundled Graphviz wasm — same pipeline as `draw_dot`, just running on the JVM kernel instead of a Python process.

In [2]:
import sk.ainet.lang.types.FP32

// n = nn.Neuron(2);  x = [Value(1.0), Value(-2.0)];  y = n(x)
val program = dag {
    val x = constant<FP32, Float>("x") {
        fromArray(floatArrayOf(1.0f, -2.0f), shape = listOf(1, 2))
    }
    val w = parameter<FP32, Float>("w") { shape(2, 1) { ones() } }
    val b = parameter<FP32, Float>("b") { shape(1) { zeros() } }

    val act = add(matmul(x, w), b)
    val y = tanh(act)                 // polyfilled as 2*sigmoid(2x)-1 (see docs/upstream/tanh-activation.md)

    output(y)
}

program.asDot()

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 8.0.5 (20230430.1635)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- const_x -->
 
 const_x 
 
 input 
 
 const_x 
 
<!-- n0_matmul -->
 
 n0_matmul 
 
 matmul 
 
 n0_matmul 
 
<!-- const_x->n0_matmul -->
 
 const_x->n0_matmul 
 
 
 
<!-- const_x_op -->
 
 const_x_op 
 
 kind: const 
 
<!-- const_x_op->const_x -->
 
 const_x_op->const_x 
 
 
 
<!-- param_w -->
 
 param_w 
 
 input 
 
 param_w 
 
<!-- param_w->n0_matmul -->
 
 param_w->n0_matmul 
 
 
 
<!-- param_w_op -->
 
 param_w_op 
 
 kind: parameter 
 
<!-- param_w_op->param_w -->
 
 param_w_op->param_w 
 
 
 
<!-- param_b -->
 
 param_b 
 
 input 
 
 param_b 
 
<!-- n1_add -->
 
 n1_add 
 
 add | n1_add 
 
<!-- param_b->n1_add -->
 
 param_b->n1_add 
 
 
 
<!-- param_b_op -->
 
 param_b_op 
 
 kind: parameter 
 
<!-- param_b_op->param_b -->
 
 param_b_op->param_b 
 
 
 
<!-- n0_matmul->n1_add -->
 
 n0_matmul->n1_add 
 
 
 
<!-- n2_mulScalar -->
 
 n2_mulScalar 
 
 mulScalar 
 
 n2_mulScalar 
 
<!-- n1_add->n2_mulScalar -->
 
 n1_add->n2_mulScalar 
 
 
 
<!-- n3_sigmoid -->
 
 n3_sigmoid 
 
 sigmoid 
 
 n3_sigmoid 
 
<!-- n2_mulScalar->n3_sigmoid -->
 
 n2_mulScalar->n3_sigmoid 
 
 
 
<!-- n2_mulScalar_op -->
 
 n2_mulScalar_op 
 
 b: 2 
 
<!-- n2_mulScalar_op->n2_mulScalar -->
 
 n2_mulScalar_op->n2_mulScalar 
 
 
 
<!-- n4_mulScalar -->
 
 n4_mulScalar 
 
 mulScalar 
 
 n4_mulScalar 
 
<!-- n3_sigmoid->n4_mulScalar -->
 
 n3_sigmoid->n4_mulScalar 
 
 
 
<!-- n5_subScalar -->
 
 n5_subScalar 
 
 subScalar 
 
 n5_subScalar 
 
<!-- n4_mulScalar->n5_subScalar -->
 
 n4_mulScalar->n5_subScalar 
 
 
 
<!-- n4_mulScalar_op -->
 
 n4_mulScalar_op 
 
 b: 2 
 
<!-- n4_mulScalar_op->n4_mulScalar -->
 
 n4_mulScalar_op->n4_mulScalar 
 
 
 
<!-- n5_subScalar_op -->
 
 n5_subScalar_op 
 
 b: 1 
 
<!-- n5_subScalar_op->n5_subScalar -->
 
 n5_subScalar_op->n5_subScalar